# Unlimited OCR Everywhere - CPU-only Laptop (experimental)

**No GPU? It runs, but slowly. Use this for a few pages, not a book.**

This notebook turns every PDF or image in `input/` into Markdown in `output/`
using Baidu's **Unlimited-OCR-2** model. It is self-contained: dependencies,
model download, PDF rendering, post-processing and figure cropping are handled
for you.

| Item | Value |
|---|---|
| **Hardware** | Any x86-64 or ARM64 CPU with 16 GB+ RAM (32 GB recommended) |
| **Model** | `baidu/Unlimited-OCR-2` (3B parameters, ~6.7 GB in BF16) |
| **First run** | Downloads ~7 GB of weights; expect 10-20 minutes on a fast connection |
| **Privacy** | Local inference. On Colab, files are processed in Google's cloud. |
| **Repo** | [https://github.com/__OWNER__/__REPO__](https://github.com/__OWNER__/__REPO__) |

## How to use

1. Install Python 3.10-3.13. For NVIDIA GPUs, install a recent driver.
2. Run the install cell, then **Run all**.
3. Put PDFs or images in `input/` (next to this notebook).
4. Run the processing cell again whenever you add more files.
5. Results appear in `output/`; a ZIP is created next to it.

The notebook uses the current working directory. If you opened it from the
repository's `notebooks/` folder, `input/` and `output/` are created there.


> **New here?** Start with the free T4 notebook. It is the lowest-friction way
> to try the model. If a document is long or you need production throughput,
> move to the L4/A100 notebook or the SGLang server notebook.


In [ ]:
#@title 1. Install dependencies (first run only)
# Pinned to the versions the model's remote code targets.
import importlib.util
import subprocess
import sys


def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


# Use the CPU-only PyTorch wheel so pip does not download a multi-GB CUDA build.
if importlib.util.find_spec("torch") is None:
    pip_install(
        "torch",
        "torchvision",
        "--index-url",
        "https://download.pytorch.org/whl/cpu",
    )

pip_install(
    "transformers==4.52.1",
    "huggingface-hub==0.36.0",
    "tokenizers==0.21.4",
    "accelerate==1.6.0",
    "addict==2.4.0",
    "einops==0.8.2",
    "easydict==1.13",
    "pymupdf==1.27.2.2",
    "pillow",
    "sentencepiece",
)

print("Python:", sys.version.split()[0])
print("Dependencies installed.")
print("If pip replaced transformers after it was already imported, restart the kernel once.")

## Configuration

The defaults below are tuned for **a CPU-only laptop**. You normally do not need
to change anything. The important knobs are:

| Setting | What it does |
|---|---|
| `MODE` | `auto` (recommended), `base` (lowest memory), `gundam` (most detail) |
| `CHUNK_PAGES` | Pages per model request. `0` = choose from available memory. |
| `DPI` | PDF render resolution. 200 is a good default; 300 is slower. |
| `MAX_PAGES` | `0` = all pages. Set a number to cap very long documents. |
| `MAX_TOKENS_PER_PAGE` | Output budget per page. Raise it if a page is cut off. |
| `FORCE_DTYPE` | `auto` picks BF16/FP16/FP32 for the device. |
| `USE_TRITON` | CUDA-only fused kernels. Disable if you see Triton errors. |

**This tier:** float32, one page per request, 150 DPI and a 5-page cap. CPU inference is experimental; a free Colab T4 is much faster.

If you change `transformers` or other packages after this notebook has imported
them, use **Runtime -> Restart runtime** (Colab) or **Kernel -> Restart**
(Jupyter), then run all cells again.


In [ ]:
#@title 2. Configuration
from pathlib import Path
import os
import sys
import json
import re
import time
import shutil
import gc
import importlib.util

# ---------------------------------------------------------------------------
# Storage
# ---------------------------------------------------------------------------
ROOT = Path.cwd()
INPUT_DIR = ROOT / "input"
OUTPUT_DIR = ROOT / "output"
REPO_DIR = ROOT.parent if (ROOT.parent / "vendor").exists() else ROOT
VENDOR_DIR = REPO_DIR / "vendor"


# ---------------------------------------------------------------------------
# Model and quality
# ---------------------------------------------------------------------------
MODEL_ID = "baidu/Unlimited-OCR-2"

# 200 dpi is plenty for most documents. 300 dpi is slower and uses more memory.
DPI = 150

# 0 = all pages. Set a number to limit very long documents.
MAX_PAGES = 5

# auto   = base for multi-page / low-VRAM, gundam for one page on a big GPU
# base   = one global view per page (lowest memory)
# gundam = tiled views (more detail, more memory)
MODE = "base"

# 0 = choose from available memory at load time. Set 1 for the lowest memory use.
CHUNK_PAGES = 1

MAX_TOKENS_PER_PAGE = 4000
MAX_PAGE_TOKENS = 20000
MAX_MODEL_LEN = 131072

# ---------------------------------------------------------------------------
# Behaviour
# ---------------------------------------------------------------------------
# Reprocess documents whose .md already exists.
FORCE = False

# Keep the rendered page JPEGs in output/.work/ (they are deleted otherwise).
KEEP_PAGES = False

# Colab only: download a ZIP when the run finishes.
AUTO_DOWNLOAD_ZIP = False

# Local only: create a ZIP next to the output folder.
MAKE_ZIP = True

# Colab: wait for uploads. Local: process whatever is already in input/.
WAIT_FOR_FILES = False
WAIT_MINUTES = 0

# auto = bfloat16 on Ampere+, float16 on T4/Turing, float32 on CPU.
# Force one with "bfloat16", "float16" or "float32".
FORCE_DTYPE = "float32"

# The model's fused Triton kernels are CUDA-only. Turn off for CPU/MPS or if
# you see Triton errors.
USE_TRITON = False

# cuda / mps / cpu / auto
REQUIRE_DEVICE = "cpu"



if MODE not in ("auto", "base", "gundam"):
    raise ValueError("MODE must be auto, base or gundam")
MAX_TOKENS_PER_PAGE = max(1, int(MAX_TOKENS_PER_PAGE))

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_README = """UNLIMITED OCR - INPUT FOLDER

Put PDF files or images in this folder (subfolders are allowed).

Supported: .pdf, .png, .jpg, .jpeg, .webp, .bmp, .tif, .tiff

Already-processed documents are skipped unless FORCE = True in the
configuration cell. Results are written to ../output/.

Tip: the first run downloads about 7 GB of model weights.
"""

OUTPUT_README = """UNLIMITED OCR - OUTPUT FOLDER

For each input file "name.pdf" (or "name.jpg") you get:

  name.md          final Markdown (figures are in name.assets/)
  name.raw.txt     raw model output (layouts, tables, boxes)
  name.meta.json   pages, timings, token counts, finish_reason
  name.assets/     cropped figure images referenced by name.md

run_summary.json is written after every batch.

You can re-run the notebook; existing .md files are skipped unless
FORCE = True.
"""

(INPUT_DIR / "README.txt").write_text(INPUT_README, encoding="utf-8")
(OUTPUT_DIR / "README.txt").write_text(OUTPUT_README, encoding="utf-8")

print("Input folder :", INPUT_DIR)
print("Output folder:", OUTPUT_DIR)
print("Device target:", REQUIRE_DEVICE)
print("Mode         :", MODE)

## Load the model

The cell below detects the device, picks a safe dtype, loads the model, and applies small compatibility patches for float16 and non-CUDA devices. It prints exactly what it chose before inference starts.

In [ ]:
#@title 3. Detect the device and load the model
import gc
import inspect
import os
import platform
import sys
import textwrap

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("SPARSE_DIAG_LOG", "0")
os.environ.setdefault("CODE_STAMP_LOG", "0")
os.environ.setdefault("RECALL_LOG", "0")

import torch


def total_ram_gb():
    try:
        import psutil

        return psutil.virtual_memory().total / 1e9
    except Exception:
        pass
    try:
        return os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
    except Exception:
        return 0.0


# ---------------------------------------------------------------------------
# Pick the device
# ---------------------------------------------------------------------------
if REQUIRE_DEVICE == "cuda":
    if not torch.cuda.is_available():
        raise RuntimeError(
            "No CUDA GPU detected. Use Runtime -> Change runtime type -> GPU "
            "(Colab) or install the CUDA-enabled PyTorch wheel (local), then "
            "re-run this notebook."
        )
    DEVICE = "cuda"
elif REQUIRE_DEVICE == "mps":
    mps_ok = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    if not mps_ok:
        print("WARNING: MPS is not available. Falling back to CPU (very slow).")
        DEVICE = "cpu"
    else:
        DEVICE = "mps"
elif REQUIRE_DEVICE == "cpu":
    DEVICE = "cpu"
else:
    if torch.cuda.is_available():
        DEVICE = "cuda"
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        DEVICE = "mps"
    else:
        DEVICE = "cpu"

if DEVICE == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {gpu_name} | {vram_gb:.1f} GB | compute {cap[0]}.{cap[1]}")
    if vram_gb < 10:
        raise RuntimeError(
            f"Only {vram_gb:.1f} GB VRAM detected. Unlimited-OCR-2 needs about "
            "12 GB minimum (the BF16 weights alone are ~6.7 GB). Use a T4/L4/A100 "
            "runtime or a larger local GPU."
        )
    if vram_gb < 12:
        print("WARNING: less than 12 GB VRAM; use MODE='base' and CHUNK_PAGES=1.")
elif DEVICE == "mps":
    gpu_name = "Apple Silicon (MPS)"
    vram_gb = 0.0
    print(f"Device: {gpu_name} | unified memory: {total_ram_gb():.1f} GB")
    print("Apple Silicon support is experimental. Close other apps before running.")
else:
    gpu_name = platform.processor() or "CPU"
    vram_gb = 0.0
    ram_gb = total_ram_gb()
    print(f"Device: CPU ({gpu_name}) | system RAM: {ram_gb:.1f} GB")
    if ram_gb and ram_gb < 16:
        print("WARNING: CPU inference needs roughly 12 GB just for float32 weights.")
        print("         Use a 16 GB+ machine, or rent a Colab T4 for free.")
    print("CPU inference is experimental and much slower than a GPU.")

# ---------------------------------------------------------------------------
# Pick the dtype
# ---------------------------------------------------------------------------
if FORCE_DTYPE != "auto":
    DTYPE = getattr(torch, FORCE_DTYPE)
    if DEVICE == "cpu" and DTYPE == torch.float16:
        print("float16 is not supported on CPU; using float32 instead.")
        DTYPE = torch.float32
elif DEVICE == "cuda":
    try:
        native_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
    except TypeError:
        native_bf16 = torch.cuda.is_bf16_supported()
    DTYPE = torch.bfloat16 if native_bf16 else torch.float16
elif DEVICE == "mps":
    DTYPE = torch.float16
else:
    DTYPE = torch.float32

print("Using dtype:", str(DTYPE).replace("torch.", ""))

# Triton kernels are CUDA-only.
USE_TRITON = bool(USE_TRITON) and DEVICE == "cuda"

# ---------------------------------------------------------------------------
# Load the tokenizer and model
# ---------------------------------------------------------------------------
from transformers import AutoConfig, AutoModel, AutoTokenizer
import transformers as _tf

print("transformers:", _tf.__version__)
if _tf.__version__ != "4.52.1":
    raise RuntimeError(
        f"transformers {_tf.__version__} is loaded, but this notebook needs 4.52.1. "
        "Restart the runtime/kernel, then run all cells again."
    )

print("Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Loading model (~6.7 GB) ...")
load_kwargs = dict(
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=DTYPE,
)
if not USE_TRITON:
    config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
    sparse_cfg = getattr(config, "sparse_attention_config", None)
    if isinstance(sparse_cfg, dict):
        sparse_cfg["dash_use_triton"] = False
        print("Triton kernels: off (dense PyTorch fallback)")
    else:
        print("WARNING: could not disable Triton in the model config")
    load_kwargs["config"] = config
else:
    print("Triton kernels: on")

try:
    model = AutoModel.from_pretrained(MODEL_ID, low_cpu_mem_usage=True, **load_kwargs)
except Exception as exc:
    print("low_cpu_mem_usage failed, retrying without it:", exc)
    gc.collect()
    model = AutoModel.from_pretrained(MODEL_ID, **load_kwargs)

model = model.eval().to(DEVICE)
torch.set_grad_enabled(False)
print(f"Model loaded. dtype={next(model.parameters()).dtype}")
if DEVICE == "cuda":
    print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


# ---------------------------------------------------------------------------
# Compatibility patches for float16/float32 and CPU/MPS
# ---------------------------------------------------------------------------
def patch_image_dtype(dtype):
    """The upstream preprocessing builds bfloat16 image tensors. Cast them."""
    patched = 0
    wanted = {type(model).__module__}
    try:
        wanted.add(type(model.model).__module__)
    except Exception:
        pass
    for name, module in list(sys.modules.items()):
        if not (name.endswith("modeling_unlimitedocr") or name in wanted):
            continue
        original = getattr(module, "_uocr_build_inputs", None)
        if original is None or getattr(original, "_uocr_dtype_patched", False):
            continue

        def wrapper(*args, __orig=original, __dtype=dtype, **kwargs):
            input_ids, images, seq_mask, spatial = __orig(*args, **kwargs)
            if images is not None:
                images = (images[0].to(__dtype), images[1].to(__dtype))
            return input_ids, images, seq_mask, spatial

        wrapper._uocr_dtype_patched = True
        module._uocr_build_inputs = wrapper
        patched += 1
    return patched


def patch_non_cuda_forward(model):
    """Upstream forward has hard-coded .cuda() calls. Rewrite them for CPU/MPS."""
    module = sys.modules.get(type(model).__module__)
    if module is None:
        return []
    patched = []
    for name, cls in list(vars(module).items()):
        if not inspect.isclass(cls) or cls.__module__ != module.__name__:
            continue
        fn = cls.__dict__.get("forward")
        if fn is None:
            continue
        try:
            src = inspect.getsource(fn)
        except (OSError, TypeError):
            continue
        if ".cuda()" not in src:
            continue
        new_src = textwrap.dedent(src).replace(
            ".cuda()", ".to(inputs_embeds.device)"
        )
        namespace = dict(module.__dict__)
        try:
            exec(compile(new_src, f"<patched {name}.forward>", "exec"), namespace)
            cls.forward = namespace["forward"]
            patched.append(name)
        except Exception as exc:
            print(f"  could not patch {name}.forward: {exc}")
    return patched


if DTYPE != torch.bfloat16:
    n = patch_image_dtype(DTYPE)
    print(f"Patched image preprocessing for {str(DTYPE).replace('torch.', '')}: {n} module(s).")

if DEVICE != "cuda":
    patched = patch_non_cuda_forward(model)
    print("Patched forward for", DEVICE + ":", ", ".join(patched) or "no .cuda() calls found")
    try:
        def _cuda_shim(self, *args, **kwargs):
            return self.to(DEVICE)

        torch.Tensor.cuda = _cuda_shim
        print("Applied torch.Tensor.cuda compatibility shim.")
    except Exception as exc:
        print("Could not apply Tensor.cuda shim (source patch should cover it):", exc)


# ---------------------------------------------------------------------------
# Pick a safe number of pages per request
# ---------------------------------------------------------------------------
if CHUNK_PAGES > 0:
    CHUNK = CHUNK_PAGES
elif DEVICE != "cuda":
    CHUNK = 1
elif vram_gb >= 40:
    CHUNK = 0          # whole document in one request
elif vram_gb >= 22:
    CHUNK = 8
elif vram_gb >= 15:
    CHUNK = 2
else:
    CHUNK = 1

print("Pages per request:", "whole document" if CHUNK == 0 else CHUNK)
print("Ready.")

In [ ]:
#@title 5. Load the post-processor
import importlib.util
import os
import urllib.request

POST_DIR = VENDOR_DIR
POST_DIR.mkdir(parents=True, exist_ok=True)
POST_PATH = POST_DIR / "process_unlimitedOCR2.py"
POST_URL = "https://raw.githubusercontent.com/baidu/Unlimited-OCR/main/utils/process_unlimitedOCR2.py"

if not POST_PATH.exists() or POST_PATH.stat().st_size < 1000:
    print("Downloading process_unlimitedOCR2.py ...")
    try:
        urllib.request.urlretrieve(POST_URL, POST_PATH)
    except Exception as exc:
        print("Download failed, trying wget:", exc)
        os.system(f"wget -q '{POST_URL}' -O '{POST_PATH}'")

if not POST_PATH.exists() or POST_PATH.stat().st_size < 1000:
    raise RuntimeError(
        "Could not download process_unlimitedOCR2.py. Download it manually from "
        f"{POST_URL} and place it at {POST_PATH}."
    )

spec = importlib.util.spec_from_file_location("process_unlimitedOCR2", POST_PATH)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
convert = module.convert

_test = convert("<PAGE><BLOCK>text[0,0,100,100]<FORK>ok")
print("Post-processor ready:", _test)

In [ ]:
#@title 6. Helper functions
import gc
import json
import re
import shutil
import time
from pathlib import Path

from PIL import Image, ImageSequence

try:
    import pymupdf
except ImportError:
    import fitz as pymupdf

INPUT_EXTS = {".pdf", ".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}
RE_IMAGE_REF = re.compile(
    r"!\[\]\(page(\d+)_(-?\d+)_(-?\d+)_(-?\d+)_(-?\d+)\.jpg\)", re.I
)


def cleanup_memory():
    gc.collect()
    if DEVICE == "cuda":
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception:
            pass
    elif DEVICE == "mps":
        try:
            torch.mps.empty_cache()
        except Exception:
            pass


def safe_doc_id(path):
    try:
        rel = path.resolve().relative_to(INPUT_DIR.resolve())
    except ValueError:
        rel = Path(path.name)
    raw = rel.with_suffix("").as_posix().replace("/", "__")
    doc_id = re.sub(r"[^\w.\-]+", "_", raw).strip("._")
    return doc_id or "document"


def list_inputs():
    return sorted(
        p
        for p in INPUT_DIR.rglob("*")
        if p.is_file()
        and p.suffix.lower() in INPUT_EXTS
        and not p.name.startswith("~$")
    )


def render_pdf(pdf, dest):
    doc = pymupdf.open(str(pdf))
    try:
        if doc.needs_pass:
            raise RuntimeError("PDF is password protected")
        total = doc.page_count
        if total == 0:
            raise RuntimeError("PDF has no pages")
        count = min(total, MAX_PAGES) if MAX_PAGES > 0 else total
        dest.mkdir(parents=True, exist_ok=True)
        pages = []
        for i in range(count):
            pix = doc[i].get_pixmap(dpi=DPI, alpha=False)
            path = dest / f"page_{i + 1:04d}.jpg"
            path.write_bytes(pix.tobytes("jpeg", jpg_quality=92))
            pages.append(path)
        return pages, total
    finally:
        doc.close()


def render_image(image_path, dest):
    dest.mkdir(parents=True, exist_ok=True)
    pages = []
    with Image.open(image_path) as src:
        for i, frame in enumerate(ImageSequence.Iterator(src)):
            if MAX_PAGES > 0 and i >= MAX_PAGES:
                break
            img = frame.convert("RGB")
            out = dest / f"page_{i + 1:04d}.jpg"
            img.save(out, "JPEG", quality=92)
            pages.append(out)
    if not pages:
        raise RuntimeError("Image has no frames")
    return pages, len(pages)


def render_document(path, dest):
    if path.suffix.lower() == ".pdf":
        return render_pdf(path, dest)
    return render_image(path, dest)


def normalize_chunk(raw):
    text = (raw or "").strip()
    while text.endswith("<PAGE>"):
        text = text[: -len("<PAGE>")].rstrip()
    if text and "<PAGE>" not in text:
        text = "<PAGE>" + text
    return text


def crop_figure_refs(markdown, pages, assets_dir):
    if not pages:
        return markdown
    assets_dir.mkdir(parents=True, exist_ok=True)
    made = 0

    def repl(match):
        nonlocal made
        page_no = int(match.group(1))
        if page_no < 1 or page_no > len(pages):
            return ""
        try:
            with Image.open(pages[page_no - 1]) as src:
                img = src.convert("RGB")
        except Exception:
            return ""
        width, height = img.size
        x1, y1, x2, y2 = (
            max(0, min(1000, int(match.group(i)))) for i in range(2, 6)
        )
        px1, py1 = int(x1 / 1000 * width), int(y1 / 1000 * height)
        px2, py2 = int(x2 / 1000 * width), int(y2 / 1000 * height)
        if px2 <= px1 or py2 <= py1:
            return ""
        name = f"page{page_no}_{x1}_{y1}_{x2}_{y2}.jpg"
        try:
            img.crop((px1, py1, px2, py2)).save(
                assets_dir / name, "JPEG", quality=90
            )
        except Exception:
            return ""
        made += 1
        return f"![]({assets_dir.name}/{name})"

    result = RE_IMAGE_REF.sub(repl, markdown)
    if made == 0:
        shutil.rmtree(assets_dir, ignore_errors=True)
    return result


def infer_chunk(paths):
    crop_mode = None if MODE == "auto" else (MODE == "gundam")
    # On small GPUs and non-CUDA devices, one global view is the safe default.
    if MODE == "auto" and len(paths) == 1:
        if DEVICE != "cuda" or (vram_gb and vram_gb < 20):
            crop_mode = False
    max_new = min(MAX_TOKENS_PER_PAGE * len(paths), MAX_MODEL_LEN - 2048)
    return model.infer(
        tokenizer=tokenizer,
        prompt="Multi page parsing.",
        image_file=[str(p) for p in paths],
        crop_mode=crop_mode,
        eval_mode=True,
        tps_interval=0 if DEVICE == "cpu" else 500,
        no_repeat_ngram_size=35,
        ngram_window=1024 if len(paths) > 1 else 128,
        page_count_guard=True,
        max_page_tokens=MAX_PAGE_TOKENS,
        max_new_tokens=max(1, max_new),
        max_length=MAX_MODEL_LEN,
        return_dict=True,
    )


def infer_with_retry(paths):
    try:
        return [infer_chunk(paths)]
    except RuntimeError as exc:
        message = str(exc).lower()
        oom = any(
            token in message
            for token in ("out of memory", "mps backend", "cuda error")
        )
        if not oom:
            raise
        cleanup_memory()
        if len(paths) > 1:
            mid = len(paths) // 2
            print(
                f"  out of memory: splitting {len(paths)} pages -> "
                f"{mid} + {len(paths) - mid}"
            )
            return infer_with_retry(paths[:mid]) + infer_with_retry(paths[mid:])
        raise RuntimeError(
            "Out of memory on a single page. Try MODE='base', lower DPI, "
            "lower MAX_TOKENS_PER_PAGE, or a machine with more memory."
        ) from exc


def process_document(path):
    started = time.time()
    doc_id = safe_doc_id(path)
    md_path = OUTPUT_DIR / f"{doc_id}.md"
    raw_path = OUTPUT_DIR / f"{doc_id}.raw.txt"
    meta_path = OUTPUT_DIR / f"{doc_id}.meta.json"
    assets_dir = OUTPUT_DIR / f"{doc_id}.assets"

    if md_path.exists() and not FORCE:
        print(f"  skip (already exists): {md_path.name}")
        return {"doc": doc_id, "status": "skipped"}

    work = OUTPUT_DIR / ".work" / doc_id
    shutil.rmtree(work, ignore_errors=True)
    work.mkdir(parents=True, exist_ok=True)

    pages, total = render_document(path, work)
    print(f"  rendered {len(pages)} page(s) from {path.name}")

    chunk_size = CHUNK if CHUNK > 0 else len(pages)
    chunks = [pages[i : i + chunk_size] for i in range(0, len(pages), chunk_size)]

    raw_parts = []
    chunk_meta = []
    truncated = False

    for index, chunk in enumerate(chunks, 1):
        print(f"  chunk {index}/{len(chunks)}: {chunk[0].stem} .. {chunk[-1].stem}")
        print("    running model (this can take several minutes) ...")
        chunk_started = time.time()
        results = infer_with_retry(chunk)
        for res in results:
            raw_parts.append(normalize_chunk(res.get("text", "")))
            chunk_meta.append(
                {
                    "pages": res.get("expected_pages"),
                    "num_pages_out": res.get("num_pages_out"),
                    "completion_tokens": len(res.get("token_ids") or []),
                    "finish_reason": res.get("finish_reason"),
                    "seconds": round(
                        res.get("time") or (time.time() - chunk_started), 1
                    ),
                }
            )
            if res.get("finish_reason") == "length":
                truncated = True
        raw_path.write_text(
            "\n".join(p for p in raw_parts if p), encoding="utf-8"
        )
        print(f"    {sum(len(p) for p in raw_parts):,} chars so far")
        cleanup_memory()

    raw = "\n".join(p for p in raw_parts if p).strip()
    if not raw:
        raise RuntimeError("The model returned an empty result.")
    raw_path.write_text(raw, encoding="utf-8")

    markdown = convert(raw)
    if not markdown.strip():
        raise RuntimeError("Post-processing returned empty Markdown.")
    shutil.rmtree(assets_dir, ignore_errors=True)
    markdown = crop_figure_refs(markdown, pages, assets_dir)
    md_path.write_text(markdown, encoding="utf-8")

    elapsed = round(time.time() - started, 1)
    meta = {
        "doc": doc_id,
        "source": str(path),
        "kind": "pdf" if path.suffix.lower() == ".pdf" else "image",
        "pages_rendered": len(pages),
        "pages_total": total,
        "dpi": DPI,
        "mode": MODE,
        "chunk_pages": CHUNK,
        "chunks": chunk_meta,
        "truncated": truncated,
        "model": MODEL_ID,
        "device": DEVICE,
        "dtype": str(DTYPE).replace("torch.", ""),
        "seconds": elapsed,
        "markdown_chars": len(markdown),
        "raw_chars": len(raw),
    }
    meta_path.write_text(
        json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    if not KEEP_PAGES:
        shutil.rmtree(work, ignore_errors=True)
        try:
            (OUTPUT_DIR / ".work").rmdir()
        except OSError:
            pass

    print(f"  saved {md_path.name} ({len(markdown):,} chars, {elapsed}s)")
    if truncated:
        print("  WARNING: output hit max_new_tokens; the last page may be cut off.")
    return {
        "doc": doc_id,
        "status": "ok",
        "pages": len(pages),
        "seconds": elapsed,
        "truncated": truncated,
    }


def run_all():
    files = list_inputs()
    if not files:
        print("No PDFs or images found in", INPUT_DIR)
        return []

    print(f"Found {len(files)} input file(s)")
    results = []
    batch_started = time.time()
    for index, path in enumerate(files, 1):
        try:
            rel = path.relative_to(INPUT_DIR)
        except ValueError:
            rel = path.name
        print(f"[{index}/{len(files)}] {rel}")
        try:
            results.append(process_document(path))
        except Exception as exc:
            print("  ERROR:", exc)
            cleanup_memory()
            results.append(
                {"doc": safe_doc_id(path), "status": "error", "error": str(exc)}
            )

    summary = {
        "model": MODEL_ID,
        "device": DEVICE,
        "dtype": str(DTYPE).replace("torch.", ""),
        "seconds": round(time.time() - batch_started, 1),
        "total": len(results),
        "ok": sum(1 for r in results if r.get("status") == "ok"),
        "failed": sum(1 for r in results if r.get("status") == "error"),
        "skipped": sum(1 for r in results if r.get("status") == "skipped"),
        "results": results,
    }
    (OUTPUT_DIR / "run_summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(
        f"\nDone in {summary['seconds']}s: "
        f"{summary['ok']} ok, {summary['skipped']} skipped, "
        f"{summary['failed']} failed"
    )
    print("Results:", OUTPUT_DIR)
    return results

## Process the input folder

Colab waits for uploads; local notebooks process whatever is already in `input/`. Re-run this cell whenever you add files.

In [ ]:
#@title 7. Process the input folder
import time

if WAIT_FOR_FILES:
    print("=" * 72)
    print("WAITING FOR FILES")
    print("Upload PDFs or images to:", INPUT_DIR)
    print("In Colab: left sidebar -> folder icon -> open that folder -> upload")
    print("=" * 72)
    deadline = time.time() + WAIT_MINUTES * 60
    last_note = 0.0
    while not list_inputs() and time.time() < deadline:
        if time.time() - last_note > 30:
            print("  still waiting ...", time.strftime("%H:%M:%S"))
            last_note = time.time()
        time.sleep(5)

files = list_inputs()
if not files:
    print("No files found. Add PDFs/images to", INPUT_DIR, "and re-run this cell.")
else:
    print(f"Found {len(files)} file(s). Starting ...")
    run_all()
    print("\nAdd more files and re-run THIS cell to process them.")

In [ ]:
#@title 8. Preview the Markdown
from IPython.display import Markdown, display

md_files = sorted(OUTPUT_DIR.glob("*.md"))
if not md_files:
    print("No Markdown files yet.")
for md_file in md_files:
    print("\n" + "=" * 72)
    print(md_file)
    print("=" * 72)
    text = md_file.read_text(encoding="utf-8", errors="replace")
    display(Markdown(text[:6000]))
    if len(text) > 6000:
        print(f"... preview truncated; open {md_file.name} for the full text.")

In [ ]:
#@title 9. Zip the results
import datetime
from pathlib import Path

if not any(OUTPUT_DIR.glob("*.md")):
    print("Nothing to zip yet.")
elif MAKE_ZIP or AUTO_DOWNLOAD_ZIP:
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_base = Path(ROOT) / f"unlimited_ocr_output_{stamp}"
    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=str(OUTPUT_DIR))
    print("ZIP:", zip_path)
    if AUTO_DOWNLOAD_ZIP:
        try:
            from google.colab import files

            files.download(zip_path)
        except Exception as exc:
            print("Automatic download failed; download the ZIP manually:", exc)
else:
    print("ZIP disabled (MAKE_ZIP = False).")

## Troubleshooting

### CPU-specific notes

- CPU inference is **experimental and very slow**. A 3B model in float32 needs
  roughly 12 GB of RAM just for the weights; 16 GB is the minimum, 32 GB is
  comfortable.
- The notebook defaults to one page per request, 150 DPI and a 5-page cap. Do
  not expect interactive speed.
- For a free GPU, use the Colab T4 notebook instead. It is dramatically faster.
- Set `OMP_NUM_THREADS` to the number of physical cores if CPU usage is low.
- If the process is killed, your machine ran out of RAM. Lower `MAX_PAGES`,
  `DPI` and `MAX_TOKENS_PER_PAGE`, or use a machine with more memory.


### The model returns empty output or cuts off

- Raise `MAX_TOKENS_PER_PAGE` (for example 8000 -> 12000).
- Check `name.meta.json`: `"truncated": true` means the output hit the budget.
- Process fewer pages per request: `CHUNK_PAGES = 1`.

### Transformers or import errors

This notebook pins `transformers==4.52.1`, the version the model's remote code
targets. Restart the runtime/kernel after installing, then run all cells.

### `CUDA out of memory`

- Set `MODE = "base"`.
- Set `CHUNK_PAGES = 1`.
- Lower `DPI` to 150-200.
- Lower `MAX_TOKENS_PER_PAGE` to 4000.
- Use a larger GPU, or open the SGLang notebook for long documents.

The notebook also retries automatically by splitting a request into smaller
page groups when it hits an out-of-memory error.

### Triton / custom kernel errors

Set `USE_TRITON = False` in the configuration cell, restart the runtime/kernel,
and run all cells again. This uses the slower pure-PyTorch attention path.

### Colab disconnected and files are gone

Set `USE_DRIVE = True` in the configuration cell, or download the ZIP before
the runtime ends. Colab's `/content` storage is temporary.

### Long documents and production use

A free Colab GPU is not meant for hundred-page documents. Use `CHUNK_PAGES` to
limit memory, or move to the SGLang server notebook, which is designed for
long-horizon parsing and concurrent requests.
